# How to solve a general eigenvalue problem

**Step 1: Read the settings from the setting file**

In [9]:
from    FELiCS.Parameters.config            import config    # Read parameters

settingsFileName = "modal.json"
param       = config()
param.importFromFile(settingsFileName)

mesh        = param.getMesh()

Info     | config.py              | importFromFile             (line 170 ) : Loading configuration from modal.json
Warning  | config.py              | importFromFile             (line 189 ) : Field "MolViscPerturbModel" missing from file, setting default: {'type': 'Constant', 'Constants': {'Viscosity': 1.0}}
Warning  | config.py              | importFromFile             (line 189 ) : Field "PrandtlNumber" missing from file, setting default: 0.72
Warning  | config.py              | importFromFile             (line 189 ) : Field "ForcingCoeff" missing from file, setting default: []
Warning  | config.py              | importFromFile             (line 189 ) : Field "ForcingNorm" missing from file, setting default: TKE
Warning  | config.py              | importFromFile             (line 189 ) : Field "ResponseNorm" missing from file, setting default: TKE
Info     | logging.py             | change_log_location        (line 263 ) : Log files moved to: output_dir/log
Info     | MixtureClass.py

**Step 2: Define FEM spaces and read in the baseflow**

We provide a baseflow file, that already contains all the necessary information of the flow field. This baseflow file gets imported and gets stored in the FELiCS meanFlowClass, which contains all the variables of the flow field.

In [ ]:
import numpy as np

from  FELiCS.SpaceDisc.FEMSpaces    import FEMSpaces
from    FELiCS.Fields.meanFlowClass         import meanFlowClass
from    FELiCS.Fields.Field                  import Field

# Get FEMSpaces
spaces   = FEMSpaces(
    param,
    mesh,
)

# Initialize mean flow class & import from file
meanFlow    = meanFlowClass(
    param, 
    spaces, 
    mesh
)
meanFlow.importDataFromFileAndExportToH5()


# baseFlow   = Field(FEMSpaces.VMixed, mesh)
baseFlow_array = meanFlow._fieldDict['u'].function.x.array[:]

# Load base flow into meanFlow object
if np.linalg.norm(meanFlow._fieldDict['u'].function.x.array[:]) < 1.e-8:
    baseFlow    = Field(spaces.VMixed, mesh)
    baseFlow.setCoefficientArray(baseFlow_array)
    [u,p]       = baseFlow.getListOfSingleFields()
    meanFlow._fieldDict['u'] = u.function

# Export mean flow in "h5" file
meanflowFilename = 'meanflow.h5'
meanFlow.mapToExportMeshAndExport(spaces, meanflowFilename)

Info     | FEMSpaces.py           | __init__                   (line 90  ) : Defining FEM-spaces.
Info     | meanFlowClass.py       | importDataFromFileAndExportToH5 (line 56  ) : Reading input flow from: 'base_flow_for_FELiCS.fel'
Info     | meanFlowClass.py       | InterpolateOnFELiCSMesh    (line 389 ) : Interpolating mean flow on FELiCS mesh.
Warning  | meanFlowClass.py       | raiseNotInFileListWarning  (line 559 ) : Field 'rho' not in import file. Assuming Field is one.
Warning  | meanFlowClass.py       | raiseNotInFileListWarning  (line 561 ) : Field 'spg' not in import file. Assuming Field is zero.
Info     | meanFlowClass.py       | mapToExportMeshAndExport   (line 146 ) : Mapping mean flow to export mesh and exporting.
Info     | meanFlowClass.py       | mapToExportMeshAndExport   (line 146 ) : Mapping mean flow to export mesh and exporting.
Info     | EquationCollection.py  | __init__                   (line 119 ) : Initializing the equation collection class.


**Step 3: Define the Equations for the linear problem**

Setting up an equation is straightforward in FELiCS. The EquationCollectionClass contains a range of predefined equations like the Navier-Stokes-Equations, that we will solve today. You can find a detailed list and information about the equations at [link]

In [ ]:
from    FELiCS.Equation.EquationCollection  import EquationCollectionClass

# Set up quations
equation    = EquationCollectionClass(
    param,
    spaces,
    meanFlow,
    mesh
)

**Step 4: Define and solve the general Eigenproblem**

The general Eigenproblem is defined by getting the linear operator and the weight matrix from the EquationCollectionClass. After initializing the `ModeCollection` class, which will store the solution of our modal analysis, we are ready to solve the general Eigenproblem.
The general Eigenproblem is solved using the `LinearSolver` class. The solution of the general Eigenproblem may take some time.
After solving the problem we can get the leading mode and its respective eigenvalue from the `ModeCollection` class. The eigenvalue printed should be approximatley 0.74+0.013j.

In [ ]:
from    FELiCS.Solvers.LinearSolver         import LinearSolver
from    FELiCS.Fields.ModeCollection        import ModeCollection

# Get matrices for eigenproblem
A       = equation.getLinearOperator(meanFlow)
B       = equation.getWeightMatrix  (meanFlow)

guesses  = param.Numerics.EigenValueGuess
nSol    = param.Numerics.nSolut
# Solve direct eigenproblem for each guess
solution    = ModeCollection(
    spaces.VMixed, 
    mesh
)
for guess in guesses:
    tmp = LinearSolver.solveGeneralEigenproblem(
        A,
        B,
        guess,
        nSol,
    )
    solution.appendSolutionOfEigenProblem(tmp, guess)

# Get leading eigenvalue
eigenValue  = solution.getLeadingMode().getEigenValue()
print(f"Leading eigenvalue: {str(eigenValue)}")

Info     | 517692396.py           | <module>                   (line 28  ) : Leading eigenvalue: (0.7447568778367061+0.013262932365237054j)
